In [1]:
from DeviceClasses.E5063A_VNA_Class import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB, dB_to_linear_voltage
from DeviceClasses.Utilities.data_management import read_file, save_data_3
import plotly.graph_objects as go
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
%matplotlib qt



In [2]:

main_dir = r".\\"
base_path = main_dir + r"circuit_2\\data\\"
S21_data_path = base_path + "\\raw_data\\"

print(os.listdir(main_dir))

['Attenuation_Processor_averager.ipynb', 'Attenuation_Processor_circuit_maker.ipynb', 'circuit_2', 'S_param_analysis.ipynb']


In [3]:
#organisation of dictionary
# SA_data = {"10[dBm]":{
#     "average_21":{
#         "freq"          :{},
#         "time"          :{},
#         "IQ_voltage_mag":{},
#     }
# }}
SA_data = {}

#load in file
for folder in os.listdir(base_path):
    S21_data_path = base_path + folder + "\\raw_data\\"

    for file in os.listdir(S21_data_path):
        path =   S21_data_path + file
        print(f"file:{file}")
        folder = S21_data_path#.split("\\")[-1]
        file_split = file.split("\\")

        data_pandas_temp, meta_data_temp = read_file(path)
        S_parameter = meta_data_temp["Result"]
    
        if not folder in SA_data.keys():
            SA_data[folder] = {}
        if not S_parameter in SA_data[folder].keys():
            SA_data[folder][S_parameter] = {}
        if not file in SA_data[folder].keys():
            SA_data[folder][S_parameter][file] = {}

        SA_data[folder][S_parameter][file] = {"data":data_pandas_temp,
                                "meta_data":meta_data_temp}
        
    plt.close()


    fig_averaged, ax_fft_averaged = plt.subplots()
    ax_fft_averaged.set(title = "S21", xlabel = "frequency[Hz]", ylabel = "mag[dB]")
    fig_averaged.tight_layout()

    #make plotly figure:
    fig_plotly_1 = go.Figure()
    fig_plotly_1.update_layout(
    title="S-parameters",
    xaxis_title="Frequency [Hz]",
    yaxis_title="Power Ratio [dB]"
    )

    S_averaged = []


    for folder, folder_dict in SA_data.items():
       
        for S_parameter, S_parameter_dict in folder_dict.items():
            print("folder:"+folder)
            for file, file_dict in S_parameter_dict.items():

                print(f"file:{file}")
                print(f"file_dict.keys:{file_dict.keys()}")

                data_pandas = file_dict["data"] 
                meta_data = file_dict["meta_data"] 

                freqs = np.array(data_pandas["freq[Hz]"])
                #print(f"freqs:{freqs}")

                powers_linear = dB_to_linear_voltage(np.array(data_pandas["sweep_number_mag[dB]:1"]))
                S_averaged.append(powers_linear)

            S_pre_averaged_linear = np.array(S_averaged)
            average = np.shape(S_averaged)[0]
            print(f"power_averaged_shape:{np.shape(S_averaged)}")
            print(f"S_averaged (list):{S_averaged}")
            print(f"S_averaged_pre:{S_pre_averaged_linear}")
            if average !=1:
                
                S_averaged_linear = np.mean(S_pre_averaged_linear, axis = 0)
                S_std_linear = np.std(S_pre_averaged_linear, axis = 0)
                S_averaged = linear_voltage_to_dB(S_averaged_linear)
                S_std_upper = linear_voltage_to_dB(S_averaged_linear + S_std_linear) - S_averaged
                S_std_lower = np.abs(linear_voltage_to_dB(S_averaged_linear - S_std_linear) - S_averaged)
                if np.any([S_averaged, S_std_upper, S_std_lower]== np.nan):
                    raise ValueError("Unwanted np.nan")
                
            else: 
                print("else")
                S_averaged_linear = S_pre_averaged_linear[0]
                S_averaged = linear_voltage_to_dB(S_averaged_linear)
                S_std_linear = np.full_like(S_averaged,None)
                S_std_upper = S_std_linear#linear_voltage_to_dB(S_averaged_linear + S_std_linear) - S_averaged
                S_std_lower = S_std_linear#linear_voltage_to_dB(S_averaged_linear - S_std_linear) - S_averaged
                
                
            print(f"S_averaged_linear:{S_averaged_linear}")
            print(f"S_averaged:{S_averaged}")
            print(f"S_std_linear:{S_std_linear}")
            print(f"S_std_upper:{S_std_upper}")
            print(f"S_std_lower:{S_std_lower}")

            file_name = f"Averaged_{S_parameter}.tsv"
            save_file_location = folder.replace("raw_data\\","")
            headers = "Frequency[Hz]\tS_parameter[dB]\tS_parameter_upper_std[dB]\tS_parameter_lower_std[dB]"
            meta_data_saved = meta_data | {"averaged":average}
            #print(S_averaged)
            #print(S_std)
            save_data = np.array([freqs, S_averaged, S_std_upper, S_std_lower])
            save_data_3(save_data, file_name, save_file_location, headers, meta_data = meta_data_saved )
            #save_data_3(save_data, file_name, file_location = "", headers = None, meta_data = {}

            ax_fft_averaged.plot(freqs, S_averaged, label = f"{S_parameter}" )
            ax_fft_averaged.legend(loc = "upper right")


            fig_plotly_1.add_trace(go.Scatter(x=freqs, y=S_averaged, mode='lines', name=f"{S_parameter}"))




            S_averaged = []
                    

        fig_averaged.show()

        file_name = f"S_parameters_PLOTLY_plot.html"
        save_path = f"{save_file_location}\\" + file_name

        fig_plotly_1.write_html(save_path)

        
    
                    

file:S21_fmin_1.000000E+09_fmax_20000000000.0_pts_10001_pwr_0_bdw_1000.0_LINear_avgs_1_1 copy.tsv
folder:.\\circuit_2\\data\\2024_04_10_Inside_Fridge_input_line\raw_data\
file:S21_fmin_1.000000E+09_fmax_20000000000.0_pts_10001_pwr_0_bdw_1000.0_LINear_avgs_1_1 copy.tsv
file_dict.keys:dict_keys(['data', 'meta_data'])
power_averaged_shape:(1, 10001)
S_averaged (list):[array([4.55469069e-03, 4.54038384e-03, 4.53847000e-03, ...,
       2.06132321e-05, 4.35754726e-05, 5.50942024e-05])]
S_averaged_pre:[[4.55469069e-03 4.54038384e-03 4.53847000e-03 ... 2.06132321e-05
  4.35754726e-05 5.50942024e-05]]
else
S_averaged_linear:[4.55469069e-03 4.54038384e-03 4.53847000e-03 ... 2.06132321e-05
 4.35754726e-05 5.50942024e-05]
S_averaged:[-46.83082222 -46.85814861 -46.86181062 ... -93.71707814 -87.21515787
 -85.177882  ]
S_std_linear:[nan nan nan ... nan nan nan]
S_std_upper:[nan nan nan ... nan nan nan]
S_std_lower:[nan nan nan ... nan nan nan]
datashape:(4, 10001)
file:S21_fmin_1.000000E+09_fmax_2000

In [4]:
a = np.std([[2, 0.018234, 0.01847, 0.01883], [1, 0.018234, 0.01847, 0.01883]], axis = 0)
print(a)

[0.5 0.  0.  0. ]


S21"

                freqs = np.array(data_pandas["freq[Hz]"])
                #print(f"freqs:{freqs}")

                powers_linear = dB_to_linear_voltage(data_pandas["sweep_number_mag[dB]:1"])
                S_averaged.append(powers_linear)


            average = np.shape(S_averaged)[0]
            print(f"power_averaged_shape:{average}")
            if average !=1:
                S_averaged = linear_voltage_to_dB(np.mean(S_averaged, axis = 0))
                S_std = linear_voltage_to_dB(np.std(S_averaged, axis = 0))
            else: 
                S_averaged = linear_voltage_to_dB(S_averaged[0])
                S_std = linear_voltage_to_dB(np.full_like(np.array(S_averaged[0]), np.nan))

            file_name = f"Averaged_{S_parameter}.tsv"
            save_file_location = folder.replace("raw_data\\","")
            headers = "Frequency[Hz]\tS_parameter[dB]"
            meta_data_saved = meta_data | {"averaged":average}
            print(S_averaged)
            print(S_std)
            save_data = np.array([freqs, S_averaged, S_std])
            save_data_3(save_data, file_name, save_file_location, headers, meta_data = meta_data_saved )
            #save_data_3(save_data, file_name, file_location = "", headers = None, meta_data = {}

            ax_fft_averaged.plot(freqs, S_averaged, label = f"{S_parameter}" )
            ax_fft_averaged.legend(loc = "upper right")


            fig_plotly_1.add_trace(go.Scatter(x=freqs, y=S_averaged, mode='lines', name=f"{S_parameter}"))




            S_averaged = []
                    

        fig_averaged.show()

        file_name = f"S_parameters_PLOTLY_plot.html"
        save_path = f"{save_file_location}\\" + file_name

        fig_plotly_1.write_html(save_path)

        
    
                    

In [5]:
print(f"data:{SA_data.keys()}")
#SA_data["10[dBm]"].pop("average_0")


data:dict_keys(['.\\\\circuit_2\\\\data\\\\2024_04_10_Inside_Fridge_input_line\\raw_data\\', '.\\\\circuit_2\\\\data\\\\2024_04_10_inside_fridge_output_line\\raw_data\\'])


In [6]:
a = np.array([np.nan]) + np.array([1])

print(a)

[nan]
